# M7 Validation: Real RGB Perception + LLM Plan + LIBERO-10 Task 0 (Linux/Colab)

End-to-end M7 acceptance on a Google Colab Linux+CUDA VM. Uses **PERCEPTION_BACKEND=real** (no sim-GT perception) with LLMPlanner, LLMFeedbackEvaluator, and Pi0.5 VLA. Target: LIBERO-10 task 0 — "put the alphabet soup can and the cream cheese box in the basket".

**Branch requirement:** run against a branch that contains the M7 changes. Set `GITHUB_BRANCH` in section 1 (do not silently reset to `main` when validating a pre-merge branch).

## 1 — Secrets & clone

Set `GITHUB_BRANCH` to the M7 feature branch **before** running the clone cell. Fresh clone and re-run paths both fetch, check out, and hard-reset to `origin/<GITHUB_BRANCH>` — never silently falling back to `origin/main`.

In [ ]:
import getpass, os, subprocess, sys

github_repo = os.environ.get('GITHUB_REPO', 'pmish11/Longor-Robotics')
github_branch = os.environ.get('GITHUB_BRANCH', '').strip()
if not github_branch:
    raise RuntimeError(
        'Set GITHUB_BRANCH to the M7 feature branch before running. '
        'Do not validate pre-merge M7 against origin/main — that silently drops M7 changes.'
    )
print(f'Using branch: {github_branch}')

os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT (repo scope): ')
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

os.chdir('/content/Longor' if os.path.exists('/content/Longor') else '/')
if '/content/Longor/src' not in sys.path:
    sys.path.insert(0, '/content/Longor/src')

if not os.path.exists('/content/Longor/.git'):
    result = subprocess.run(
        ['git', 'clone',
         f"https://{os.environ['GITHUB_TOKEN']}@github.com/{github_repo}.git",
         '/content/Longor'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stderr.replace(os.environ['GITHUB_TOKEN'], '***'))
        raise RuntimeError('git clone failed')
    print('Cloned OK')

os.chdir('/content/Longor')
if '/content/Longor/src' not in sys.path:
    sys.path.insert(0, '/content/Longor/src')

for cmd in (
    ['git', 'fetch', 'origin', github_branch],
    ['git', 'checkout', github_branch],
    ['git', 'reset', '--hard', f'origin/{github_branch}'],
):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/Longor')
    print(result.stdout or result.stderr or 'ok')
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'{" ".join(cmd)} failed')

## 2 — App deps + LIBERO + headless

In [ ]:
import importlib.util
import logging
import os
import subprocess
import sys

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r requirements-libero.txt

spec = importlib.util.find_spec('robosuite')
if spec:
    script = spec.submodule_search_locations[0] + '/scripts/setup_macros.py'
    result = subprocess.run([sys.executable, script], check=False)
    if result.returncode != 0:
        logging.warning('robosuite setup_macros.py failed (rc=%s); continuing', result.returncode)

## 3 — VLA venv + checkpoint

In [ ]:
import glob, os, shutil, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip')

if not os.path.exists('/root/openpi'):
    subprocess.run(
        ['git', 'clone', '--recurse-submodules', 'https://github.com/Physical-Intelligence/openpi.git', '/root/openpi'],
        check=True,
    )
else:
    print('/root/openpi already exists, skipping.')
vla_pip('install', '-e', '/root/openpi')
vla_pip('install', '-r', '/content/Longor/services/requirements-vla-pi05-linux.txt')

CKPT_CACHE = '/root/.cache/openpi/openpi-assets/checkpoints'
LIBERO_CKPT = os.path.join(CKPT_CACHE, 'pi05_libero')

for partial in glob.glob(f'{CKPT_CACHE}/*.partial'):
    shutil.rmtree(partial)
    print(f'Cleared partial: {partial}')

if os.path.isdir(LIBERO_CKPT):
    print(f'Checkpoint already present: {LIBERO_CKPT}')
else:
    os.makedirs(CKPT_CACHE, exist_ok=True)
    %pip install --quiet crcmod
    print('Downloading pi05_libero checkpoint via gsutil...')
    !gsutil -m -o GSUtil:num_retries=5 cp -r gs://openpi-assets/checkpoints/pi05_libero {CKPT_CACHE}
    print('Download complete.')

## 4 — Env config (M7: PERCEPTION_BACKEND=real)

In [ ]:
import importlib.util, os, yaml

env_content = f"""OPENAI_API_KEY={os.environ['OPENAI_API_KEY']}
VLA_POLICY=pi05
VLA_MAX_TOOL_SECONDS=180
OPENPI_CONFIG=pi05_libero
BACKEND=libero
PERCEPTION_BACKEND=real
LLM_MODEL=gpt-4.1-mini
LONGOR_TRACE_DIR=traces
LONGOR_MAX_REPLANS=3
LONGOR_STEP_BUDGET=30
VLA_PORT=8765
VLA_GRIPPER_CONFIDENCE=0.5
VLA_GRIPPER_MODE=absolute_open_close
VLA_ACTION_SCALE=1.0
VLA_STEPS_PER_ACTION=1
VLA_ADAPTIVE_SCALE=false
VLA_CHUNK_STEPS=5
VLA_STAGNATION_CHUNKS=20
LIBERO_SETTLE_STEPS=10
LIBERO_EE_DELTA_SCALE=1.0
VLA_MAX_INNER_STEPS=200
VLA_FAIL_DIST_CONSECUTIVE_STEPS=200
VLA_FAIL_IK_SATURATION_STEPS=200
VLA_FAIL_WORSENING_STEPS=200
VLA_USE_ENV_TASK_SUCCESS=false
VLA_PROXIMITY_GRASP=false
LIBERO_PREPARE_GRAB_AFTER_PLACE=1
LIBERO_PREPARE_GRAB_XY_RETREAT=1
LIBERO_PREPARE_GRAB_XY_RETREAT_STEPS=15
LIBERO_PREPARE_GRAB_XY_RETREAT_DELTA=0.15
LIBERO_PREPARE_GRAB_DESCEND_DZ=0.35
LIBERO_PREPARE_GRAB_DESCEND_STEPS=20
LIBERO_PREPARE_GRAB_TARGET_Z=0.55
LIBERO_PREPARE_GRAB_LOW_Z_MAX_STEPS=30
LIBERO_PREPARE_GRAB_APPROACH_MAX_STEPS=80
LIBERO_PREPARE_GRAB_APPROACH_DELTA=0.15
LIBERO_PREPARE_GRAB_APPROACH_RADIUS=0.15
LIBERO_PLACE_SMOKE_MAX_XY=0.12
"""
with open('/content/Longor/.env', 'w') as f:
    f.write(env_content)
print('Wrote /content/Longor/.env (PERCEPTION_BACKEND=real)')

libero_config_path = os.environ.get('LIBERO_CONFIG_PATH', os.path.expanduser('~/.libero'))
config_file = os.path.join(libero_config_path, 'config.yaml')

if not os.path.exists(config_file):
    os.makedirs(libero_config_path, exist_ok=True)
    spec = importlib.util.find_spec('libero')
    libero_pkg = os.path.join(os.path.dirname(spec.origin), 'libero')
    default_config = {
        'benchmark_root': libero_pkg,
        'bddl_files': os.path.join(libero_pkg, 'bddl_files'),
        'init_states': os.path.join(libero_pkg, 'init_files'),
        'datasets': os.path.join(os.path.dirname(libero_pkg), 'datasets'),
        'assets': os.path.join(libero_pkg, 'assets'),
    }
    with open(config_file, 'w') as f:
        yaml.dump(default_config, f)
    print(f'Created {config_file}')
else:
    print(f'Already exists: {config_file}')

## 5 — Start VLA server

In [ ]:
import json, os, subprocess, time, urllib.request

VLA_LOG = '/tmp/m7_vla_pi05_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'

def _tail_log(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'

subprocess.run(['fuser', '-k', '8765/tcp'], capture_output=True)

log = open(VLA_LOG, 'w', buffering=1)
env_for_server = {**os.environ, 'OPENPI_CONFIG': 'pi05_libero'}
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor',
    env=env_for_server,
)

POLL_SEC = 15
MAX_WAIT_SEC = 20 * 60
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    elapsed = (i + 1) * POLL_SEC

    if vla_proc.poll() is not None:
        print(_tail_log(60))
        raise RuntimeError(f'VLA server exited early (code {vla_proc.returncode}). Log above.')

    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('VLA server ready:', json.loads(resp.read()))
        break
    except Exception:
        print(f'Waiting... ({elapsed}s)')
        if elapsed % 60 == 0:
            print('--- recent log ---')
            print(_tail_log(15))
else:
    print(_tail_log(60))
    raise RuntimeError(f'VLA server did not come up in {MAX_WAIT_SEC // 60} minutes. Log above.')

## 6 — M7 perception wiring smoke (unit tests, no sim-GT guard)

In [ ]:
import os, subprocess, sys
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_perception_smoke_{stamp}.txt'

result = subprocess.run(
    [
        sys.executable, '-m', 'pytest',
        'tests/test_libero_real_perception.py',
        'tests/test_libero_backend.py::test_libero_backend_register_skills_real_mode',
        'tests/test_libero_backend.py::test_libero_backend_register_skills',
        'tests/test_libero_backend.py::test_libero_backend_list_objects_routes_real_perception',
        'tests/test_libero_backend.py::test_libero_backend_list_objects_real_mode_no_detection_returns_empty',
        '-m', 'not integration', '-v',
    ],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'},
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(combined[-6000:])
print('Exit code:', result.returncode)
assert result.returncode == 0, f'M7 perception smoke failed (exit {result.returncode})'
print('M7 perception wiring smoke PASS')

## 7 — Run LLM example (real perception)

In [ ]:
import os, subprocess, sys
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_llm_{stamp}.txt'
run_log_path = f'{trace_dir}/m7_llm_{stamp}.log'

result = subprocess.run(
    [sys.executable, 'examples/long_horizon_llm_libero.py'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=1200,
    env={
        **os.environ,
        'PYTHONPATH': '/content/Longor/src',
        'MUJOCO_GL': 'egl',
        'PYOPENGL_PLATFORM': 'egl',
        'PERCEPTION_BACKEND': 'real',
        'LONGOR_RUN_LOG': run_log_path,
    },
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(f'Run log: {run_log_path}')
print('=== Key lines (non-vla.tools DEBUG) ===')
for line in combined.splitlines():
    if line.startswith('DEBUG [vla.tools]'):
        continue
    print(line)
print('Exit code:', result.returncode)
assert result.returncode == 0, f'exit code {result.returncode}'
assert 'ImportError' not in combined, 'Import failure detected'
assert 'ModuleNotFoundError' not in combined, 'Module not found'
_log_source = combined
if os.path.exists(run_log_path):
    with open(run_log_path) as _f:
        _log_source = combined + '\n' + _f.read()
assert 'Task succeeded:' in _log_source, (
    'E2E finished but task was not marked succeeded — task not achieved'
)
assert 'Run summary:' in _log_source and 'success=True' in _log_source, (
    'Expected Run summary with success=True'
)

## 8 — Trace + pytest + teardown

In [ ]:
import glob, json, os

trace_dir = '/content/Longor/traces'
files = sorted(glob.glob(f'{trace_dir}/*.jsonl'))
if not files:
    print('No trace files found')
else:
    print(f'Latest trace: {files[-1]}')
    with open(files[-1]) as f:
        events = [json.loads(l) for l in f if l.strip()]
    print(f'Total events: {len(events)}')
    for e in events:
        evt = e.get('event')
        skill = e.get('skill', '')
        step = e.get('step', '')
        if evt == 'plan_start':
            skills = e.get('skills')
            if skills:
                print(f"Plan skills: {' → '.join(skills)}")
        if skill in ('arm.grab', 'arm.place'):
            r = {}
            try:
                r = json.loads(e.get('result', '{}'))
            except Exception:
                pass
            print(f"[step {step}] {skill}: status={r.get('status')} reason={r.get('reason')} steps_applied={r.get('steps_applied')} min_dist={r.get('min_dist')} last_raw={r.get('last_raw_action')}")
        elif evt == 'skill_call' and skill not in ('arm.grab', 'arm.place'):
            print(f"[step {step}] {skill}: {str(e.get('result',''))[:120]}")
        elif evt == 'feedback':
            reason = e.get('reason')
            reason_part = f' — {reason}' if reason else ''
            print(f"[step {step}] feedback {e.get('description','')!r} → {e.get('decision')}{reason_part}")
        elif evt in ('replan', 'plan_end'):
            print(json.dumps(e, indent=2))

run_logs = sorted(glob.glob(f'{trace_dir}/m7_*.log'))
if run_logs:
    latest_log = run_logs[-1]
    print(f'Latest M7 run log: {latest_log}')
    with open(latest_log) as f:
        print(f.read()[-4000:])
else:
    print('No m7_*.log files found under traces')

In [ ]:
import os, subprocess, sys
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_pytest_{stamp}.txt'

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-m', 'not integration', '--tb=short', '-q'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'},
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(combined[-4000:])
assert result.returncode == 0, f'pytest failed (exit {result.returncode})'

vla_proc.terminate()
print('VLA server stopped.')